# Imports

In [1]:
import os
import json
import random
import shutil
import tempfile
import torch
from huggingface_hub import login, HfApi, hf_hub_download
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

In [2]:
!pip install -q tokenizers 
!pip install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/149.8 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 5.3 MB/s eta 0:00:00


In [3]:
import re
import ijson
from collections import Counter
from tokenizers import ByteLevelBPETokenizer

## Configuration


In [4]:
# =========================
# PROJECT CONFIGURATION
# =========================
from pathlib import Path
# Hugging Face
HF_USERNAME = "A-240031"
HF_REPO_NAME = "legal-lm-phase1"
HF_MODEL_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
HF_DATASET_REPO_ID = "Exploration-Lab/CS-781-Capstone"
# Dataset files
DATASET_FILES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

# Checkpointing
CHECKPOINT_INTERVAL_TOKENS = 1_000_000

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


#TOKENIZER CONFIGURATION
VOCAB_SIZE = 32_768
MIN_FREQUENCY = 5
CUSTOM_TOKEN_MIN_FREQUENCY = 100

SPECIAL_TOKENS = [
    "<|pad|>",
    "<|unk|>",
    "<|bos|>",
    "<|eos|>",
]

RCH_TRAIN_TOKENS = 50_000_000

VAL_EXAMPLES = 500
VAL_PROMPT_WORDS = 512
VAL_TARGET_WORDS = 128
VAL_DOC_FRACTION = 0.01

SEED = 42

ARCH_TRAIN_PATH = "/kaggle/working/architecture_data/arch_train_mixed.bin"
ARCH_VAL_PATH = "/kaggle/working/architecture_data/arch_val_mixed.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

In [5]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
BUCKET_NAME = "A-240031/legal-lm-phase1-bucket"
BUCKET_ROOT = f"hf://buckets/{BUCKET_NAME}"

from huggingface_hub import HfFileSystem

fs = HfFileSystem()

In [7]:
def upload_to_bucket(local_path, remote_path):
    remote_path = remote_path.lstrip("/")

    fs.put_file(
        local_path,
        f"{BUCKET_ROOT}/{remote_path}",
    )

    print(f"Uploaded: {remote_path}")

In [8]:
def download_from_bucket(remote_path, local_path):
    remote_path = remote_path.lstrip("/")

    fs.get_file(
        f"{BUCKET_ROOT}/{remote_path}",
        local_path,
    )

    print(f"Downloaded: {remote_path} -> {local_path}")

## Checkpoint related functions

In [9]:
def upload_checkpoint(tmp_dir, tag):

    remote_dir = f"checkpoints/{tag}"

    for filename in os.listdir(tmp_dir):
        local_path = os.path.join(tmp_dir, filename)

        upload_to_bucket(
            local_path,
            f"{remote_dir}/{filename}",
        )

In [10]:
def save_checkpoint(tag, model, optimizer, scheduler=None, scaler=None,
                    tokens_seen=0, global_step=0, epoch=0, extra_state=None):

    with tempfile.TemporaryDirectory() as tmp_dir:

        torch.save(model.state_dict(), os.path.join(tmp_dir, "model.pt"))
        torch.save(optimizer.state_dict(), os.path.join(tmp_dir, "optimizer.pt"))

        if scheduler is not None:
            torch.save(scheduler.state_dict(), os.path.join(tmp_dir, "scheduler.pt"))

        if scaler is not None:
            torch.save(scaler.state_dict(), os.path.join(tmp_dir, "scaler.pt"))

        rng_state = {"python": random.getstate(), "torch": torch.get_rng_state()}

        if torch.cuda.is_available():
            rng_state["cuda"] = torch.cuda.get_rng_state_all()

        torch.save(rng_state, os.path.join(tmp_dir, "rng_state.pt"))

        state = {
            "tag": tag,
            "tokens_seen": tokens_seen,
            "global_step": global_step,
            "epoch": epoch,
            "extra_state": extra_state,
        }

        with open(os.path.join(tmp_dir, "training_state.json"), "w") as f:
            json.dump(state, f, indent=2)
        upload_checkpoint(tmp_dir,tag)
        # HfApi().upload_folder(
        #     repo_id=HF_MODEL_REPO_ID,
        #     repo_type="model",
        #     folder_path=tmp_dir,
        #     path_in_repo=f"checkpoints/{tag}",
        #     commit_message=f"Save checkpoint: {tag}",
        # )

def load_checkpoint(tag, model, optimizer, scheduler=None, scaler=None):

    with tempfile.TemporaryDirectory() as tmp_dir:

        files = [
            "model.pt",
            "optimizer.pt",
            "rng_state.pt",
            "training_state.json",
        ]

        checkpoint_dir = os.path.join(
            tmp_dir,
            "checkpoints",
            tag,
        )

        for filename in files:
            try:
                download_from_bucket(
                    f"checkpoints/{tag}/{filename}",
                    os.path.join(checkpoint_dir, filename),
                )
            except Exception:
                pass

        model.load_state_dict(
            torch.load(
                os.path.join(checkpoint_dir, "model.pt"),
                map_location=DEVICE,
                weights_only=True,
            )
        )

        optimizer.load_state_dict(
            torch.load(
                os.path.join(checkpoint_dir, "optimizer.pt"),
                map_location=DEVICE,
                weights_only=False,
            )
        )
        rng_state = torch.load(
            os.path.join(checkpoint_dir, "rng_state.pt"),
            map_location="cpu",
            weights_only=False,
        )

        random.setstate(rng_state["python"])
        torch.set_rng_state(rng_state["torch"])

        if torch.cuda.is_available() and "cuda" in rng_state:
            torch.cuda.set_rng_state_all(rng_state["cuda"])

        with open(
            os.path.join(checkpoint_dir, "training_state.json")
        ) as f:
            state = json.load(f)

    return state

## Loading training data

In [11]:
MIXED_DATASET_PATH = "/kaggle/working/mixed_200m.jsonl"

download_from_bucket(
    "architecture_data/mixed_200m.jsonl",
    MIXED_DATASET_PATH,
)

(…)ucket/architecture_data/mixed_200m.jsonl:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Downloaded: architecture_data/mixed_200m.jsonl -> /kaggle/working/mixed_200m.jsonl


In [12]:
def corpus_iterator(path):
    import json

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            text = record.get("text")
            if text:
                yield text

In [13]:
tokenizer = ByteLevelBPETokenizer(add_prefix_space=False,lowercase=False)
tokenizer.train_from_iterator(corpus_iterator(MIXED_DATASET_PATH),vocab_size=VOCAB_SIZE,min_frequency=MIN_FREQUENCY,special_tokens=SPECIAL_TOKENS)

In [14]:
def add_custom_legal_tokens(tokenizer, corpus_path, min_frequency=100):
    patterns = [
        re.compile(r"\b(?:Article|ARTICLE)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\b(?:Section|SECTION)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\b(?:Rule|RULE)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\b(?:Regulation|REGULATION)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\([a-z]\)"),
        re.compile(r"\([0-9]{1,2}\)"),
        re.compile(r"\((?:i{1,3}|iv|v|vi{0,3}|ix|x{1,2})\)"),
    ]

    counts = Counter()

    for text in corpus_iterator(corpus_path):
        for pattern in patterns:
            counts.update(pattern.findall(text))

    custom_tokens = []

    for token, count in counts.items():
        token_with_space = "Ġ" + token

        if count >= min_frequency and tokenizer.token_to_id(token_with_space) is None:
            custom_tokens.append(token_with_space)

    tokenizer.add_tokens(custom_tokens)

    return custom_tokens, counts

In [15]:
custom_tokens, custom_counts = add_custom_legal_tokens(tokenizer,MIXED_DATASET_PATH,CUSTOM_TOKEN_MIN_FREQUENCY)

In [16]:
VOCAB_SIZE = tokenizer.get_vocab_size()

## Splitting dataset into train and val set for choosing architecture

In [17]:
def load_or_create_architecture_data(mixed_path):

    try:
        download_from_bucket(
                "architecture_data/arch_train_mixed.bin",
                ARCH_TRAIN_PATH,
            )
            
        download_from_bucket(
                "architecture_data/arch_val_mixed.json",
                ARCH_VAL_PATH,
            )

        print("Mixed architecture data loaded from Hugging Face.")
        return

    except Exception:
        print("Mixed architecture data not found. Creating from mixed_200m...")

    

    # Count documents for document-wise split
    num_docs = 0

    with open(mixed_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)

            if record.get("text"):
                num_docs += 1

    split = int(num_docs * VAL_DOC_FRACTION)

    print(f"Total documents: {num_docs:,}")
    print(f"Validation documents: {split:,}")
    print(f"Training documents: {num_docs - split:,}")

    val_examples = []

    with open(mixed_path, "r", encoding="utf-8") as src, \
         open(ARCH_TRAIN_PATH, "wb") as train_file:

        doc_idx = 0

        for line in src:

            record = json.loads(line)
            text = record.get("text")

            if not text:
                continue

            # -------------------------
            # Validation documents
            # -------------------------
            if doc_idx < split:

                words = text.split()

                if len(words) >= VAL_PROMPT_WORDS + VAL_TARGET_WORDS:

                    for start in range(
                        0,
                        len(words)
                        - VAL_PROMPT_WORDS
                        - VAL_TARGET_WORDS
                        + 1,
                        VAL_PROMPT_WORDS,
                    ):

                        val_examples.append({
                            "prompt": " ".join(
                                words[
                                    start:start + VAL_PROMPT_WORDS
                                ]
                            ),
                            "target": " ".join(
                                words[
                                    start + VAL_PROMPT_WORDS:
                                    start + VAL_PROMPT_WORDS + VAL_TARGET_WORDS
                                ]
                            ),
                        })

                        if len(val_examples) >= VAL_EXAMPLES:
                            break

            # -------------------------
            # Training documents
            # -------------------------
            else:

                ids = tokenizer.encode(text).ids

                ids.append(
                    tokenizer.token_to_id("<|eos|>")
                )

                np.asarray(
                    ids,
                    dtype=np.uint16
                ).tofile(train_file)

            doc_idx += 1

    # Save validation examples
    with open(
        ARCH_VAL_PATH,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            val_examples,
            f,
            ensure_ascii=False
        )

    # Upload mixed architecture training data
    upload_to_bucket(ARCH_TRAIN_PATH,"architecture_data/arch_train_mixed.bin" )

    # Upload mixed architecture validation data
    upload_to_bucket(ARCH_VAL_PATH,"architecture_data/arch_val_mixed.json" )
    print("Mixed architecture data created and uploaded.")

In [18]:
load_or_create_architecture_data(MIXED_DATASET_PATH)


(…)t/architecture_data/arch_train_mixed.bin:   0%|          | 0.00/249M [00:00<?, ?B/s]

Downloaded: architecture_data/arch_train_mixed.bin -> /kaggle/working/architecture_data/arch_train_mixed.bin


(…)et/architecture_data/arch_val_mixed.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Downloaded: architecture_data/arch_val_mixed.json -> /kaggle/working/architecture_data/arch_val_mixed.json
Mixed architecture data loaded from Hugging Face.


In [19]:
import numpy as np
import torch


def loader(
    path,
    batch_size=4,
    encoder_len=512,
    target_len=128,
):
    tokens = np.memmap(
        path,
        dtype=np.uint16,
        mode="r",
    )

    seq_len = encoder_len + target_len

    batch_encoder = []
    batch_target = []

    for start in range(
        0,
        len(tokens) - seq_len + 1,
        seq_len,
    ):
        chunk = tokens[start:start + seq_len]

        batch_encoder.append(
            torch.from_numpy(
                chunk[:encoder_len].copy()
            ).long()
        )

        batch_target.append(
            torch.from_numpy(
                chunk[encoder_len:].copy()
            ).long()
        )

        if len(batch_encoder) == batch_size:

            yield (
                torch.stack(batch_encoder),
                torch.stack(batch_target),
            )

            batch_encoder = []
            batch_target = []

    if batch_encoder:
        yield (
            torch.stack(batch_encoder),
            torch.stack(batch_target),
        )


In [20]:
train_loader = loader(
    path=ARCH_TRAIN_PATH,
    batch_size=4,
    encoder_len=512,
    target_len=128,
)

In [21]:
import json
import torch


def val_loader(
    path,
    batch_size=4,
    encoder_len=512,
    target_len=128,
):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    batch_encoder = []
    batch_target = []

    for record in data:
        prompt_ids = tokenizer.encode(
            record["prompt"]
        ).ids[:encoder_len]

        target_ids = tokenizer.encode(
            record["target"]
        ).ids[:target_len]

        if len(prompt_ids) != encoder_len:
            continue

        if len(target_ids) != target_len:
            continue

        batch_encoder.append(
            torch.tensor(prompt_ids, dtype=torch.long)
        )

        batch_target.append(
            torch.tensor(target_ids, dtype=torch.long)
        )

        if len(batch_encoder) == batch_size:
            yield (
                torch.stack(batch_encoder),
                torch.stack(batch_target),
            )

            batch_encoder = []
            batch_target = []

    if batch_encoder:
        yield (
            torch.stack(batch_encoder),
            torch.stack(batch_target),
        )

In [22]:
def get_val_data():
    return val_loader(
        ARCH_VAL_PATH,
        batch_size=4,
        encoder_len=512,
        target_len=128,
    )

## Model - A2

In [23]:
DIM = 640
ENCODER_LAYERS = 6
DECODER_LAYERS = 6
N_HEADS = 16
N_KV_HEADS = 4          # GQA
FF_DIM = 1728
MAX_ENCODER_LEN = 512
MAX_DECODER_LEN = 256
RMS_NORM_EPS = 1e-6
ROPE_BASE = 10000
GRAD_CLIP = 5.0

In [24]:
# ============================================================
# RMSNorm
# ============================================================

class RMSNorm(nn.Module):

    def __init__(self, dim, eps=RMS_NORM_EPS):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )

        self.eps = eps

    def forward(self, x):

        rms = (x*x).mean(
            dim=-1,
            keepdim=True
        )

        return self.weight * x * torch.rsqrt(
            rms + self.eps
        )


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):

    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=ROPE_BASE,
    ):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2,
                    dtype=torch.float32,
                ) / head_dim
            )
        )

        positions = torch.arange(
            max_seq_len,
            dtype=torch.float32,
        )

        freqs = torch.outer(
            positions,
            inv_freq,
        )

        # [T, head_dim]
        emb = torch.cat(
            [freqs, freqs],
            dim=-1,
        )

        self.register_buffer(
            "cos",
            emb.cos()[None, None, :, :],
            persistent=False,
        )

        self.register_buffer(
            "sin",
            emb.sin()[None, None, :, :],
            persistent=False,
        )

    def forward(self, seq_len):

        return (
            self.cos[:, :, :seq_len],
            self.sin[:, :, :seq_len],
        )


def rotate_half(x):

    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]

    return torch.cat(
        [-x2, x1],
        dim=-1,
    )


def apply_rope(q, k, cos, sin):

    q = (
        q * cos
        +
        rotate_half(q) * sin
    )

    k = (
        k * cos
        +
        rotate_half(k) * sin
    )

    return q, k


# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):

    def __init__(
        self,
        dim,
        hidden_dim,
    ):
        super().__init__()

        self.gate_proj = nn.Linear(
            dim,
            hidden_dim,
            bias=False,
        )

        self.up_proj = nn.Linear(
            dim,
            hidden_dim,
            bias=False,
        )

        self.down_proj = nn.Linear(
            hidden_dim,
            dim,
            bias=False,
        )

    def forward(self, x):

        return self.down_proj(
            F.silu(
                self.gate_proj(x)
            )
            *
            self.up_proj(x)
        )


# ============================================================
# GQA + Flash Attention
# ============================================================

class Attention(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        max_seq_len,
        causal=False,
        use_rope=True,
    ):
        super().__init__()

        assert dim % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.dim = dim
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads

        self.head_dim = dim // n_heads

        self.causal = causal
        self.use_rope = use_rope

        # ----------------------------------------------------
        # Q / K / V projections
        # ----------------------------------------------------

        self.q_proj = nn.Linear(
            dim,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            dim,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            dim,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.o_proj = nn.Linear(
            n_heads * self.head_dim,
            dim,
            bias=False,
        )

        # RoPE only needs to be constructed for self-attention
        if use_rope:

            self.rope = RotaryEmbedding(
                self.head_dim,
                max_seq_len,
            )

    def forward(
        self,
        x,
        kv=None,
        past_kv=None,
        use_cache=False,
    ):

        B, T, C = x.shape

        # ====================================================
        # Queries
        # ====================================================

        q = self.q_proj(x)

        q = q.view(
            B,
            T,
            self.n_heads,
            self.head_dim,
        )

        q = q.transpose(1, 2)

        # ====================================================
        # Keys / Values
        # ====================================================

        # Self-attention:
        #     source = x
        #
        # Cross-attention:
        #     source = encoder output

        source = x if kv is None else kv

        S = source.shape[1]

        k = self.k_proj(source)
        v = self.v_proj(source)

        k = k.view(
            B,
            S,
            self.n_kv_heads,
            self.head_dim,
        )

        v = v.view(
            B,
            S,
            self.n_kv_heads,
            self.head_dim,
        )

        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # ====================================================
        # RoPE
        # ====================================================

        # Only self-attention gets RoPE.
        # Cross-attention does NOT use RoPE.

        if kv is None and self.use_rope:

            current_len = k.shape[2]

            cos, sin = self.rope(
                current_len
            )

            q, k = apply_rope(
                q,
                k,
                cos,
                sin,
            )

        # ====================================================
        # Decoder KV Cache
        # ====================================================

        if (
            kv is None
            and past_kv is not None
        ):

            past_k, past_v = past_kv

            k = torch.cat(
                [past_k, k],
                dim=2,
            )

            v = torch.cat(
                [past_v, v],
                dim=2,
            )

        # ====================================================
        # GQA
        # ====================================================

        if self.n_kv_heads != self.n_heads:

            repeat_factor = (
                self.n_heads
                // self.n_kv_heads
            )

            k = k.repeat_interleave(
                repeat_factor,
                dim=1,
            )

            v = v.repeat_interleave(
                repeat_factor,
                dim=1,
            )

        # ====================================================
        # Flash / SDPA Attention
        # ====================================================

        # During cached decoding, only one new token is
        # queried, so causal masking is unnecessary.

        is_causal = (
            self.causal
            and kv is None
            and past_kv is None
        )

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=0.0,
            is_causal=is_causal,
        )

        # ====================================================
        # Output projection
        # ====================================================

        out = out.transpose(
            1,
            2,
        ).contiguous()

        out = out.view(
            B,
            T,
            C,
        )

        out = self.o_proj(out)

        # Cache only decoder self-attention
        new_kv = None

        if use_cache and kv is None:
            new_kv = (k, v)

        return out, new_kv


# ============================================================
# Encoder Block
# ============================================================

class FNetEncoderBlock(nn.Module):
    def __init__(self, dim, ff_dim):
        super().__init__()

        self.norm1 = RMSNorm(dim)
        self.norm2 = RMSNorm(dim)

        self.ffn = SwiGLU(dim, ff_dim)

    def forward(self, x):
        # Fourier token mixing
        x_norm = self.norm1(x)

        # 2D FFT over sequence and hidden dimensions
        x_fft = torch.fft.fft2(x_norm, dim=(-2, -1)).real

        x = x + x_fft

        # FFN
        x = x + self.ffn(self.norm2(x))

        return x

# ============================================================
# Decoder Block
# ============================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        ff_dim,
        max_self_attn_len,
        max_cross_attn_len,
    ):
        super().__init__()

        # ----------------------------------------------------
        # Causal self-attention
        # ----------------------------------------------------

        self.norm1 = RMSNorm(dim)

        self.self_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_self_attn_len,
            causal=True,
            use_rope=True,
        )

        # ----------------------------------------------------
        # Encoder-decoder cross-attention
        # ----------------------------------------------------

        self.norm2 = RMSNorm(dim)

        self.cross_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_cross_attn_len,
            causal=False,
            use_rope=False,
        )

        # ----------------------------------------------------
        # FFN
        # ----------------------------------------------------

        self.norm3 = RMSNorm(dim)

        self.ffn = SwiGLU(
            dim,
            ff_dim,
        )

    def forward(
        self,
        x,
        encoder_output,
        past_kv=None,
        use_cache=False,
    ):

        # ====================================================
        # 1. Decoder causal self-attention
        # ====================================================

        residual = x

        x_norm = self.norm1(x)

        self_out, new_kv = self.self_attn(
            x_norm,
            past_kv=past_kv,
            use_cache=use_cache,
        )

        x = residual + self_out

        # ====================================================
        # 2. Cross-attention
        # ====================================================

        residual = x

        x_norm = self.norm2(x)

        cross_out, _ = self.cross_attn(
            x_norm,
            kv=encoder_output,
        )

        x = residual + cross_out

        # ====================================================
        # 3. FFN
        # ====================================================

        x = x + self.ffn(
            self.norm3(x)
        )

        return x, new_kv


# ============================================================
# Encoder-Decoder Language Model
# ============================================================

class EncoderDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        dim=DIM,
        encoder_layers=ENCODER_LAYERS,
        decoder_layers=DECODER_LAYERS,
        n_heads=N_HEADS,
        n_kv_heads=N_KV_HEADS,
        ff_dim=FF_DIM,
        max_encoder_len=MAX_ENCODER_LEN,
        max_decoder_len=MAX_DECODER_LEN,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim

        # ====================================================
        # Shared token embedding
        # ====================================================

        self.token_embedding = nn.Embedding(
            vocab_size,
            dim,
        )

        # ====================================================
        # Encoder
        # ====================================================

        self.encoder_blocks = nn.ModuleList([
            FNetEncoderBlock(
                dim=dim,
                ff_dim=ff_dim,
            )
            for _ in range(encoder_layers)
        ])

        self.encoder_norm = RMSNorm(dim)

        # ====================================================
        # Decoder
        # ====================================================

        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(
                dim=dim,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                ff_dim=ff_dim,
                max_self_attn_len=256,
                max_cross_attn_len=max_encoder_len,
            )
            for _ in range(decoder_layers)
        ])

        self.decoder_norm = RMSNorm(dim)

        # ====================================================
        # Output head
        # ====================================================

        self.lm_head = nn.Linear(
            dim,
            vocab_size,
            bias=False,
        )

        # Tie output projection to token embedding
        self.lm_head.weight = (
            self.token_embedding.weight
        )

    # ========================================================
    # ENCODER
    # ========================================================

    def encode(self, encoder_ids):

        x = self.token_embedding(
            encoder_ids
        )

        for block in self.encoder_blocks:
            x = block(x)

        x = self.encoder_norm(x)

        return x

    # ========================================================
    # DECODER
    # ========================================================

    def decode(
        self,
        decoder_ids,
        encoder_output,
        past_kv=None,
        use_cache=False,
    ):

        x = self.token_embedding(
            decoder_ids
        )

        new_cache = []

        for i, block in enumerate(
            self.decoder_blocks
        ):

            layer_past = None

            if past_kv is not None:
                layer_past = past_kv[i]

            x, layer_cache = block(
                x=x,
                encoder_output=encoder_output,
                past_kv=layer_past,
                use_cache=use_cache,
            )

            if use_cache:
                new_cache.append(
                    layer_cache
                )

        x = self.decoder_norm(x)

        logits = self.lm_head(x)

        if use_cache:
            return logits, new_cache

        return logits, None

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(
        self,
        encoder_ids,
        decoder_ids,
        labels=None,
    ):

        # Encode 512-token context
        encoder_output = self.encode(
            encoder_ids
        )

        # Decode continuation
        logits, _ = self.decode(
            decoder_ids,
            encoder_output,
        )

        # No labels -> inference
        if labels is None:
            return logits

        # ====================================================
        # Language-model loss
        # ====================================================

        loss = F.cross_entropy(
            logits.reshape(
                -1,
                self.vocab_size,
            ),
            labels.reshape(-1),
            ignore_index=-100,
        )

        return logits, loss

    # ========================================================
    # GENERATION
    # ========================================================

    @torch.no_grad()
    def generate(
        self,
        encoder_ids,
        bos_id,
        eos_id=None,
        max_new_tokens=128,
        temperature=1.0,
        top_p=0.9,
    ):

        self.eval()

        # ----------------------------------------------------
        # Encode context ONCE
        # ----------------------------------------------------

        encoder_output = self.encode(
            encoder_ids
        )

        # ----------------------------------------------------
        # Start decoder
        # ----------------------------------------------------

        B = encoder_ids.size(0)

        decoder_ids = torch.full(
            (B, 1),
            bos_id,
            dtype=torch.long,
            device=encoder_ids.device,
        )

        past_kv = None

        # ----------------------------------------------------
        # Autoregressive generation
        # ----------------------------------------------------

        for step in range(max_new_tokens):

            # Only feed the newest token once cache exists
            if past_kv is None:
                current_ids = decoder_ids
            else:
                current_ids = decoder_ids[:, -1:]

            logits, past_kv = self.decode(
                decoder_ids=current_ids,
                encoder_output=encoder_output,
                past_kv=past_kv,
                use_cache=True,
            )

            next_logits = logits[:, -1, :]

            # Temperature
            if temperature != 1.0:
                next_logits = (
                    next_logits / temperature
                )

            # ------------------------------------------------
            # Top-p sampling
            # ------------------------------------------------

            probs = F.softmax(
                next_logits,
                dim=-1,
            )

            sorted_probs, sorted_indices = torch.sort(
                probs,
                descending=True,
                dim=-1,
            )

            cumulative_probs = torch.cumsum(
                sorted_probs,
                dim=-1,
            )

            remove = (
                cumulative_probs > top_p
            )

            # Keep at least the first token
            remove[:, 0] = False

            sorted_probs[remove] = 0.0

            sorted_probs = (
                sorted_probs
                /
                sorted_probs.sum(
                    dim=-1,
                    keepdim=True,
                )
            )

            next_token = torch.multinomial(
                sorted_probs,
                num_samples=1,
            )

            next_token = torch.gather(
                sorted_indices,
                -1,
                next_token,
            )

            decoder_ids = torch.cat(
                [
                    decoder_ids,
                    next_token,
                ],
                dim=1,
            )

            if (
                eos_id is not None
                and torch.all(
                    next_token == eos_id
                )
            ):
                break

        # Remove BOS
        return decoder_ids[:, 1:]


# ============================================================
# Parameter count
# ============================================================



In [25]:
model = EncoderDecoderLM(
    vocab_size=VOCAB_SIZE,
    dim=DIM,
    encoder_layers=ENCODER_LAYERS,
    decoder_layers=DECODER_LAYERS,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    ff_dim=FF_DIM,
    max_encoder_len=MAX_ENCODER_LEN,
    max_decoder_len=MAX_DECODER_LEN,
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.1,
)

In [26]:
import time
import torch
import torch.nn.functional as F


def train(
    model,
    train_loader,
    optimizer,
    train_tokens,
    tag,
    val_interval_tokens=1_000_000,
    resume=False,
    resume_tag=None,
):
    model.train()

    tokens_seen = 0
    global_step = 0
    epoch = 0
    next_val_tokens = val_interval_tokens

    # --------------------------------------------------------
    # Resume
    # --------------------------------------------------------

    if resume:

        if resume_tag is None:
            raise ValueError(
                "resume_tag must be provided when resume=True"
            )

        state = load_checkpoint(
            tag=resume_tag,
            model=model,
            optimizer=optimizer,
        )

        tokens_seen = state["tokens_seen"]
        global_step = state["global_step"]
        epoch = state["epoch"]

        next_val_tokens = (
            (tokens_seen // val_interval_tokens) + 1
        ) * val_interval_tokens

        print(
            f"Resumed from {resume_tag} | "
            f"Tokens: {tokens_seen:,} | "
            f"Step: {global_step}"
        )

    start_time = time.perf_counter()

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    while tokens_seen < train_tokens:

        for encoder_ids, target_ids in train_loader:

            if tokens_seen >= train_tokens:
                break

            encoder_ids = encoder_ids.to(
                DEVICE,
                non_blocking=True,
            )

            target_ids = target_ids.to(
                DEVICE,
                non_blocking=True,
            )

            # ------------------------------------------------
            # Decoder input:
            #
            # [last prompt token, B1, B2, ..., B127]
            #
            # Labels:
            #
            # [B1, B2, B3, ..., B128]
            # ------------------------------------------------

            decoder_ids = torch.cat(
                [
                    encoder_ids[:, -1:],
                    target_ids[:, :-1],
                ],
                dim=1,
            )

            # ------------------------------------------------
            # Forward
            # ------------------------------------------------

            optimizer.zero_grad(
                set_to_none=True
            )

            encoder_output = model.encode(
                encoder_ids
            )

            logits, _ = model.decode(
                decoder_ids,
                encoder_output,
            )

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    model.vocab_size,
                ),
                target_ids.reshape(-1),
            )

            # ------------------------------------------------
            # Backward
            # ------------------------------------------------

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )

            optimizer.step()

            # ------------------------------------------------
            # Token accounting
            # ------------------------------------------------

            batch_tokens = (
                encoder_ids.numel()
                + target_ids.numel()
            )

            tokens_seen += batch_tokens
            global_step += 1

            # ------------------------------------------------
            # Timing
            # ------------------------------------------------

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            elapsed = (
                time.perf_counter()
                - start_time
            )

            tokens_per_sec = (
                tokens_seen / elapsed
            )

            print(
                f"Step {global_step:5d} | "
                f"Tokens {tokens_seen:,}/{train_tokens:,} | "
                f"Loss {loss.item():.4f} | "
                f"Time {elapsed:.2f}s | "
                f"Tokens/s {tokens_per_sec:,.0f}"
            )

            # =================================================
            # VALIDATION + CHECKPOINT
            # =================================================

            if (
                tokens_seen >= next_val_tokens
            ):

                model.eval()

                val_loss = 0.0
                val_tokens = 0

                with torch.no_grad():

                    for val_encoder_ids,val_target_ids in get_val_data():

                        val_encoder_ids = val_encoder_ids.to(
                            DEVICE,
                            non_blocking=True,
                        )

                        val_target_ids = val_target_ids.to(
                            DEVICE,
                            non_blocking=True,
                        )

                        # Same teacher-forcing setup
                        val_decoder_ids = torch.cat(
                            [
                                val_encoder_ids[:, -1:],
                                val_target_ids[:, :-1],
                            ],
                            dim=1,
                        )

                        val_encoder_output = model.encode(
                            val_encoder_ids
                        )

                        val_logits, _ = model.decode(
                            val_decoder_ids,
                            val_encoder_output,
                        )

                        loss = F.cross_entropy(
                            val_logits.reshape(
                                -1,
                                model.vocab_size,
                            ),
                            val_target_ids.reshape(-1),
                            reduction="sum",
                        )

                        val_loss += loss.item()
                        val_tokens += val_target_ids.numel()

                val_loss /= val_tokens

                print(
                    f"Validation @ {tokens_seen:,} tokens | "
                    f"Val Loss: {val_loss:.4f}"
                )

                model.train()

                # ------------------------------------------------
                # Save checkpoint
                # ------------------------------------------------

                save_checkpoint(
                    tag=tag,
                    model=model,
                    optimizer=optimizer,
                    tokens_seen=tokens_seen,
                    global_step=global_step,
                    epoch=epoch,
                )

                print(
                    f"Checkpoint saved: {tag}\n"
                )

                next_val_tokens += val_interval_tokens

        epoch += 1

    # --------------------------------------------------------
    # Complete
    # --------------------------------------------------------

    total_time = (
        time.perf_counter()
        - start_time
    )

    print("\nTraining complete.")
    print(
        f"Tokens seen: {tokens_seen:,}"
    )
    print(
        f"Total time: {total_time:.2f}s"
    )
    print(
        f"Average tokens/s: "
        f"{tokens_seen / total_time:,.0f}"
    )

In [27]:
train(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    train_tokens=200_000_000,
    tag="encoder_decoder_mixed_ffn",
    val_interval_tokens=1_000_000,
)

Step     1 | Tokens 2,560/200,000,000 | Loss 597.3248 | Time 1.20s | Tokens/s 2,142
Step     2 | Tokens 5,120/200,000,000 | Loss 552.0615 | Time 1.38s | Tokens/s 3,717


Step     3 | Tokens 7,680/200,000,000 | Loss 457.9742 | Time 1.55s | Tokens/s 4,956
Step     4 | Tokens 10,240/200,000,000 | Loss 295.6960 | Time 1.71s | Tokens/s 5,974


Step     5 | Tokens 12,800/200,000,000 | Loss 182.8551 | Time 1.88s | Tokens/s 6,815
Step     6 | Tokens 15,360/200,000,000 | Loss 130.2060 | Time 2.04s | Tokens/s 7,517


Step     7 | Tokens 17,920/200,000,000 | Loss 99.3133 | Time 2.21s | Tokens/s 8,117
Step     8 | Tokens 20,480/200,000,000 | Loss 85.5565 | Time 2.37s | Tokens/s 8,632


Step     9 | Tokens 23,040/200,000,000 | Loss 78.5970 | Time 2.54s | Tokens/s 9,082
Step    10 | Tokens 25,600/200,000,000 | Loss 72.2385 | Time 2.70s | Tokens/s 9,479


Step    11 | Tokens 28,160/200,000,000 | Loss 68.6621 | Time 2.87s | Tokens/s 9,827
Step    12 | Tokens 30,720/200,000,000 | Loss 67.1400 | Time 3.03s | Tokens/s 10,132


... (25,908 outputs left out)

Step 48525 | Tokens 124,224,000/200,000,000 | Loss 1.8755 | Time 13214.00s | Tokens/s 9,401
Step 48526 | Tokens 124,226,560/200,000,000 | Loss 3.7649 | Time 13214.19s | Tokens/s 9,401


Step 48527 | Tokens 124,229,120/200,000,000 | Loss 3.4291 | Time 13214.38s | Tokens/s 9,401
Step 48528 | Tokens 124,231,680/200,000,000 | Loss 3.4809 | Time 13214.56s | Tokens/s 9,401


Step 48529 | Tokens 124,234,240/200,000,000 | Loss 1.9715 | Time 13214.75s | Tokens/s 9,401
Step 48530 | Tokens 124,236,800/200,000,000 | Loss 1.6388 | Time 13214.93s | Tokens/s 9,401


Step 48531 | Tokens 124,239,360/200,000,000 | Loss 2.5755 | Time 13215.12s | Tokens/s 9,401
Step 48532 | Tokens 124,241,920/200,000,000 | Loss 2.6129 | Time 13215.31s | Tokens/s 9,401


In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

In [ ]:
MAX_GENERATION_TOKENS = 256

In [ ]:
BATCH_SIZE = 8

test_items = list(test_data.items())

test_batches = [
    test_items[i:i + BATCH_SIZE]
    for i in range(0, len(test_items), BATCH_SIZE)
]

print("Number of test examples:", len(test_items))
print("Number of batches:", len(test_batches))

In [ ]:
@torch.no_grad()
def generate_batch(
    prompts,
    model,
    tokenizer,
    device,
    target_words=128,
    temperature=0.8,
    top_p=0.9,
):
    model.eval()

    # --------------------------------------------------------
    # Tokenize prompts
    # --------------------------------------------------------

    prompt_ids = [
        tokenizer.encode(prompt).ids[:MAX_ENCODER_LEN]
        for prompt in prompts
    ]

    input_ids = torch.tensor(
        prompt_ids,
        dtype=torch.long,
        device=device,
    )

    batch_size = input_ids.size(0)

    # --------------------------------------------------------
    # Encode all prompts ONCE
    # --------------------------------------------------------

    encoder_output = model.encode(input_ids)

    # Last prompt token is the first decoder input
    decoder_ids = input_ids[:, -1:]

    generated_ids = [
        [] for _ in range(batch_size)
    ]

    finished = torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=device,
    )

    # --------------------------------------------------------
    # Autoregressive generation
    # --------------------------------------------------------

    for _ in range(MAX_GENERATION_TOKENS):

        logits, _ = model.decode(
            decoder_ids,
            encoder_output,
        )

        next_logits = (
            logits[:, -1, :]
            / temperature
        )

        # ----------------------------------------------------
        # Top-p sampling
        # ----------------------------------------------------

        sorted_logits, sorted_indices = torch.sort(
            next_logits,
            descending=True,
            dim=-1,
        )

        sorted_probs = torch.softmax(
            sorted_logits,
            dim=-1,
        )

        cumulative_probs = torch.cumsum(
            sorted_probs,
            dim=-1,
        )

        remove = cumulative_probs > top_p

        remove[:, 1:] = remove[:, :-1].clone()
        remove[:, 0] = False

        sorted_logits[remove] = float("-inf")

        probs = torch.softmax(
            sorted_logits,
            dim=-1,
        )

        sampled = torch.multinomial(
            probs,
            1,
        )

        next_token = torch.gather(
            sorted_indices,
            -1,
            sampled,
        )

        # ----------------------------------------------------
        # Store tokens
        # ----------------------------------------------------

        for i in range(batch_size):

            if not finished[i]:
                generated_ids[i].append(
                    next_token[i].item()
                )

        # ----------------------------------------------------
        # Next decoder input
        # ----------------------------------------------------

        decoder_ids = torch.cat(
            [
                decoder_ids,
                next_token,
            ],
            dim=1,
        )

        # ----------------------------------------------------
        # Stop examples independently once they reach
        # target_words
        # ----------------------------------------------------

        for i in range(batch_size):

            if not finished[i]:

                text = tokenizer.decode(
                    generated_ids[i]
                )

                if len(text.split()) >= target_words:
                    finished[i] = True

        if finished.all():
            break

    # --------------------------------------------------------
    # Final predictions
    # --------------------------------------------------------

    predictions = []

    for ids in generated_ids:

        text = tokenizer.decode(ids)

        text = " ".join(
            text.split()[:target_words]
        )

        predictions.append(text)

    return predictions

In [ ]:
predictions = {}

for batch in test_batches:

    batch_ids = [x[0] for x in batch]
    batch_prompts = [x[1]["text"] for x in batch]

    batch_predictions = generate_batch(
        prompts=batch_prompts,
        model=model,
        tokenizer=tokenizer,
        device=DEVICE,
        target_words=128,
        temperature=0.8,
        top_p=0.9,
    )

    for test_id, prediction in zip(
        batch_ids,
        batch_predictions,
    ):
        predictions[test_id] = {
            "text": prediction
        }

In [ ]:
import zipfile
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("prediction.json saved:")
print(PREDICTION_PATH)

with zipfile.ZipFile(
    SUBMISSION_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:
    zipf.write(
        PREDICTION_PATH,
        arcname="prediction.json"
    )

print("Codabench submission created:")
print(SUBMISSION_ZIP)